# BERT虚假新闻检测模型

本 Notebook 使用普通 BERT 模型完成虚假新闻二分类任务。

数据集读取方式：
- `fake-and-real-news-dataset--Fake.xlsx`
- `fake-and-real-news-dataset--True.xlsx`

标签设置：
- `0`：虚假新闻
- `1`：真实新闻


In [ ]:
# ======================
# 1. 导入依赖库
# ======================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from transformers import BertTokenizer, TFBertModel
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model

print("TensorFlow版本：", tf.__version__)


In [ ]:
# ======================
# 2. 固定随机种子
# ======================

seed = 42

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

print("随机种子已固定：", seed)


In [ ]:
# ======================
# 3. 读取数据集
# ======================

fake_path = 'E:/Fakenews Detection/Dataest_2.0/fake-and-real-news-dataset--Fake.xlsx'
true_path = 'E:/Fakenews Detection/Dataest_2.0/fake-and-real-news-dataset--True.xlsx'

fake_df = pd.read_excel(fake_path, engine='openpyxl')
true_df = pd.read_excel(true_path, engine='openpyxl')

print("虚假新闻数据量：", len(fake_df))
print("真实新闻数据量：", len(true_df))

fake_df.head()


In [ ]:
# ======================
# 4. 添加标签并合并数据
# ======================

# 0表示虚假新闻，1表示真实新闻
fake_df['label'] = 0
true_df['label'] = 1

data_df = pd.concat([fake_df, true_df], ignore_index=True)

# 检查text列是否存在
if 'text' not in data_df.columns:
    raise ValueError("数据集中未找到 text 列，请检查Excel文件字段名称。")

# 清理文本
data_df['text'] = data_df['text'].fillna('').astype(str)

# 删除空文本
data_df = data_df[data_df['text'].str.strip() != ''].reset_index(drop=True)

print("合并后数据量：", len(data_df))
print(data_df['label'].value_counts())

data_df.head()


In [ ]:
# ======================
# 5. 划分训练集和测试集
# ======================

X = data_df['text'].values
y = data_df['label'].values.astype(float)

X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("训练集数量：", len(X_train_texts))
print("测试集数量：", len(X_test_texts))
print("训练集标签分布：", np.bincount(y_train.astype(int)))
print("测试集标签分布：", np.bincount(y_test.astype(int)))


In [ ]:
# ======================
# 6. 加载BERT Tokenizer并编码文本
# ======================

max_len = 128

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def encode_texts(texts, max_len=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='tf'
    )

train_encodings = encode_texts(X_train_texts, max_len=max_len)
test_encodings = encode_texts(X_test_texts, max_len=max_len)

print("训练集input_ids形状：", train_encodings['input_ids'].shape)
print("测试集input_ids形状：", test_encodings['input_ids'].shape)


In [ ]:
# ======================
# 7. 构建普通BERT二分类模型
# ======================

bert_model = TFBertModel.from_pretrained('bert-base-uncased')

def build_bert_classifier(max_len=128):
    input_ids = Input(shape=(max_len,), dtype=tf.int32, name='input_ids')
    attention_mask = Input(shape=(max_len,), dtype=tf.int32, name='attention_mask')

    bert_outputs = bert_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    # 取[CLS]向量作为整篇新闻文本表示
    cls_output = bert_outputs.last_hidden_state[:, 0, :]

    x = Dense(256, activation='relu')(cls_output)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid', name='output')(x)

    model = Model(
        inputs=[input_ids, attention_mask],
        outputs=output
    )

    return model

model = build_bert_classifier(max_len=max_len)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
# ======================
# 8. 训练模型
# ======================

history = model.fit(
    {
        'input_ids': train_encodings['input_ids'],
        'attention_mask': train_encodings['attention_mask']
    },
    y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=8,
    verbose=1
)


In [ ]:
# ======================
# 9. 测试集预测
# ======================

y_pred_prob = model.predict(
    {
        'input_ids': test_encodings['input_ids'],
        'attention_mask': test_encodings['attention_mask']
    }
).ravel()

y_pred = (y_pred_prob >= 0.5).astype(int)

print("预测完成")


In [ ]:
# ======================
# 10. 输出评价指标
# ======================

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\n分类报告：")
print(classification_report(y_test, y_pred, target_names=['Fake', 'True'], zero_division=0))

print("混淆矩阵：")
print(confusion_matrix(y_test, y_pred))


In [ ]:
# ======================
# 11. 保存预测结果
# ======================

result_df = pd.DataFrame({
    'text': X_test_texts,
    'true_label': y_test.astype(int),
    'pred_label': y_pred.astype(int),
    'pred_probability': y_pred_prob
})

save_path = 'E:/Fakenews Detection/Dataest_2.0/bert_prediction_results.xlsx'
result_df.to_excel(save_path, index=False)

print("预测结果已保存至：", save_path)
result_df.head()


In [ ]:
# ======================
# 12. 保存模型
# ======================

model_save_dir = 'E:/Fakenews Detection/Dataest_2.0/bert_fake_news_model'

model.save(model_save_dir)

print("模型已保存至：", model_save_dir)


In [ ]:
# ======================
# 13. 单条新闻测试
# ======================

def predict_news(text):
    text = str(text)
    encoded = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='tf'
    )

    prob = model.predict({
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask']
    })[0][0]

    pred = 1 if prob >= 0.5 else 0

    if pred == 1:
        label_text = '真实新闻'
        confidence = prob
    else:
        label_text = '虚假新闻'
        confidence = 1 - prob

    print("检测结果：", label_text)
    print("置信度：", round(float(confidence), 4))
    print("真实新闻概率：", round(float(prob), 4))
    print("虚假新闻概率：", round(float(1 - prob), 4))

    return pred, prob

# 示例
sample_text = "The government announced a new policy today."
predict_news(sample_text)
